# Fase 1 · Problema y entorno de trabajo
**Airbnb en Santiago: Grupo 4 · MCDI500**

## Introducción
El proyecto describe qué ubicaciones, configuraciones y comodidades se asocian
con mayor ocupación e ingresos estimados en Airbnb Santiago. Su propósito es
orientar la comparación de perfiles para una eventual compra, sin estimar rentabilidad.

Los objetivos son comparar el desempeño por comuna, por tamaño y por comodidades,
y reconocer las limitaciones de estas asociaciones. F1 documenta el problema,
la procedencia y las condiciones de ejecución; F2 prepara los datos, F3 verifica
las transformaciones y F4 presenta los resultados.

### Herramientas
Python y pandas permiten revisar los datos; Jupyter reúne código y resultados.
El entorno virtual y las versiones fijas permiten repetir la ejecución. Git
registra los cambios y GitHub conserva el trabajo compartido.

In [1]:
import sys                        # el intérprete en uso: se verifica mas abajo
import json                       # exportar metadatos legibles
import platform                   # el sistema operativo, para la bitácora
import subprocess                 # consultas a Git desde el cuaderno
import shutil                     # ubicar ejecutables (git) sin asumir que existen
import importlib                  # importar el módulo que este cuaderno escribe
import importlib.util             # comprobar si un paquete está instalado
from pathlib import Path          # rutas que sirven igual en cualquier sistema
from datetime import date

# Revisión de las dependencias del kernel.
_p = Path(sys.executable).parts
print("Intérprete del kernel:", Path(*_p[_p.index(".venv"):]) if ".venv" in _p else sys.executable)

REQUERIDOS = {"numpy": "numpy", "pandas": "pandas"}   # modulo: paquete pip
faltantes = [pkg for mod, pkg in REQUERIDOS.items()
             if importlib.util.find_spec(mod) is None]

if faltantes:
    raise ImportError("Instale requirements.txt en el entorno del kernel.")

import numpy as np                # acá solo para verificar el entorno
import pandas as pd               # las tablas de documentación de la fase

print("numpy", np.__version__, "· pandas", pd.__version__)

# Reproducibilidad: la misma semilla del cuaderno de la Fase 2 y del
# muestreo estratificado de src/generar_muestra.py.
SEMILLA = 42
np.random.seed(SEMILLA)

print("Cuaderno de la Fase 1 ·", date.today().isoformat())

Intérprete del kernel: .venv/bin/python


numpy 2.5.4 · pandas 3.1.1
Cuaderno de la Fase 1 · 2026-11-24


## 1. Definición del problema
La configuración reúne la pregunta, los objetivos, el alcance y los supuestos del proyecto.

In [2]:
PROYECTO = {
    "titulo": "Análisis de alojamientos Airbnb en Santiago de Chile para la decisión de inversión inmobiliaria",
    "grupo": "Grupo 4",
    "asignatura": "MCDI500 · Programación para la Ciencia de Datos",
    "integrantes": [
        "Alfredo Valenzuela Aburto",
        "Mauricio Martínez Quiñones",
        "María Laura Lanosa",
    ],
    "problematica": (
        "Un inversionista que está evaluando comprar un departamento para arriendo "
        "turístico en Santiago no tiene evidencia sistemática sobre qué "
        "ubicación, tipología y equipamiento se asocian a mayores ingresos. El "
        "proyecto describe esa relación a partir de 18.904 alojamientos "
        "publicados en Airbnb (Inside Airbnb, captura del 25–26 de septiembre "
        "de 2026), de los que se toma una muestra estratificada por comuna."
    ),
    "pregunta_investigacion": (
        "¿Qué características de ubicación, tipología, capacidad y servicios "
        "presentan los alojamientos completos activos en Airbnb Santiago con "
        "mayor ocupación e ingresos estimados?"
    ),
    "objetivo_general": (
        "Describir la relación entre ubicación, configuración y comodidades "
        "de los alojamientos completos activos en Airbnb Santiago y su "
        "desempeño comercial estimado (ocupación e ingresos), para identificar "
        "perfiles que orienten una decisión de compra."
    ),
    "objetivos_especificos": [
        "Comparar ocupación, precio e ingresos estimados entre comunas, considerando dispersión y tamaño de muestra.",
        "Describir las diferencias de desempeño según dormitorios, baños y capacidad anunciada.",
        "Examinar la asociación entre comodidades declaradas y cuartil superior de ocupación, considerando confusión.",
        "Delimitar perfiles de interés y sus limitaciones para una decisión de inversión, sin estimar rentabilidad.",
    ],
    "preguntas": [
        "¿Cómo se distribuyen el precio por noche y la ocupación estimada entre las 32 comunas?",
        "¿Cómo varían el precio y la ocupación estimada según el número de dormitorios, baños y capacidad?",
        "¿Qué comodidades son más frecuentes entre los alojamientos del cuartil superior de ocupación estimada, en comparación con el resto?",
    ],
    "criterios_exito": [
        "El repositorio se clona y los cuadernos corren completos sin intervención manual.",
        "Cada decisión de preprocesamiento queda justificada con cifras, no con adjetivos.",
        "Los integrantes tienen commits propios en el historial.",
    ],
    "alcance": {
        "incluye": [
            "Departamentos y casas completas (room_type = Entire home/apt)",
            "Alojamientos activos: reseña en los últimos 12 meses y disponibilidad futura",
            "Análisis descriptivo de ocupación estimada, precio e ingresos estimados",
        ],
        "excluye": [
            "Rentabilidad o retorno de la inversión (no hay datos de costos)",
            "Modelos predictivos o causales",
            "Análisis de sentimiento y comparación entre ciudades",
        ],
    },
    "limitaciones": [
        "Sesgo de supervivencia: solo salen los anuncios que seguían publicados en septiembre de 2026; los que cerraron no están.",
        "La ocupación estimada sale de las reseñas y tiene un tope de 255 noches (~70 % del año); no es un registro de reservas.",
        "Los ingresos estimados son ocupación × precio publicado: heredan ambos sesgos y no son una métrica independiente.",
        "Los cuadernos trabajan sobre una muestra estratificada del archivo detallado, no sobre el universo completo.",
    ],
}

La función presenta la definición del proyecto y verifica sus campos obligatorios.

In [3]:
def presentar_proyecto(config):
    """
    Imprime la definicion del proyecto de forma legible.

    Parametros
    ----------
    config : dict
        Diccionario de configuracion del proyecto.

    Retorna
    -------
    None

    Lanza
    -----
    KeyError
        Si falta alguna de las claves obligatorias.
    """
    obligatorias = ("titulo", "problematica", "objetivo_general", "objetivos_especificos")
    faltantes = [c for c in obligatorias if c not in config]
    if faltantes:
        # Se falla temprano y el mensaje dice QUE falta, no solo que algo fallo.
        raise KeyError(f"Faltan claves obligatorias en la configuracion: {faltantes}")

    print(config["titulo"].upper())
    print("=" * 76)
    print(f"{config['asignatura']} · {config['grupo']}")
    print("\nProblemática")
    print(config["problematica"])
    print("\nPregunta de investigación")
    print(config["pregunta_investigacion"])
    print("\nObjetivo general")
    print(config["objetivo_general"])
    print("\nObjetivos específicos")
    for i, obj in enumerate(config["objetivos_especificos"], start=1):
        print(f"  {i}. {obj}")
    print("\nPreguntas operativas que orientan el trabajo")
    for pregunta in config["preguntas"]:
        print(f"  · {pregunta}")


presentar_proyecto(PROYECTO)

ANÁLISIS DE ALOJAMIENTOS AIRBNB EN SANTIAGO DE CHILE PARA LA DECISIÓN DE INVERSIÓN INMOBILIARIA
MCDI500 · Programación para la Ciencia de Datos · Grupo 4

Problemática
Un inversionista que está evaluando comprar un departamento para arriendo turístico en Santiago no tiene evidencia sistemática sobre qué ubicación, tipología y equipamiento se asocian a mayores ingresos. El proyecto describe esa relación a partir de 18.904 alojamientos publicados en Airbnb (Inside Airbnb, captura del 25–26 de septiembre de 2026), de los que se toma una muestra estratificada por comuna.

Pregunta de investigación
¿Qué características de ubicación, tipología, capacidad y servicios presentan los alojamientos completos activos en Airbnb Santiago con mayor ocupación e ingresos estimados?

Objetivo general
Describir la relación entre ubicación, configuración y comodidades de los alojamientos completos activos en Airbnb Santiago y su desempeño comercial estimado (ocupación e ingresos), para identificar perfiles

El alcance delimita el análisis. Las restricciones y los supuestos se declaran por separado.

In [4]:
print("Alcance de la Fase 1 y del proyecto")
print("  Incluye:")
for item in PROYECTO["alcance"]["incluye"]:
    print(f"    + {item}")
print("  Excluye:")
for item in PROYECTO["alcance"]["excluye"]:
    print(f"    - {item}")

print("\nSesgos y limitaciones declaradas")
for lim in PROYECTO["limitaciones"]:
    print(f"  · {lim}")

Alcance de la Fase 1 y del proyecto
  Incluye:
    + Departamentos y casas completas (room_type = Entire home/apt)
    + Alojamientos activos: reseña en los últimos 12 meses y disponibilidad futura
    + Análisis descriptivo de ocupación estimada, precio e ingresos estimados
  Excluye:
    - Rentabilidad o retorno de la inversión (no hay datos de costos)
    - Modelos predictivos o causales
    - Análisis de sentimiento y comparación entre ciudades

Sesgos y limitaciones declaradas
  · Sesgo de supervivencia: solo salen los anuncios que seguían publicados en septiembre de 2026; los que cerraron no están.
  · La ocupación estimada sale de las reseñas y tiene un tope de 255 noches (~70 % del año); no es un registro de reservas.
  · Los ingresos estimados son ocupación × precio publicado: heredan ambos sesgos y no son una métrica independiente.
  · Los cuadernos trabajan sobre una muestra estratificada del archivo detallado, no sobre el universo completo.


La captura representa anuncios observados en un solo momento. Se consideran el
sesgo de supervivencia, la ocupación estimada desde reseñas y la censura en 255 noches.

## 2. Entorno
Se comprueba el intérprete, el entorno virtual y las versiones instaladas.

In [5]:
DEPENDENCIAS = ["numpy", "pandas", "matplotlib", "seaborn", "scipy"]

# Correspondencia entre el nombre de importación y el nombre del paquete en PyPI:
# Se registran los nombres de los paquetes declarados.
NOMBRE_EN_PYPI = {"scipy": "scipy"}


def verificar_entorno(dependencias):
    """
    Comprueba interprete, entorno virtual, carpeta de trabajo y librerias.

    Parametros
    ----------
    dependencias : list of str
        Nombres de importacion de las librerias que el proyecto declara.

    Retorna
    -------
    dict
        Resultado de cada comprobacion, para dejarlo en la bitacora de la fase.
    """
    reporte = {}

    # 1. El intérprete que corre ESTE cuaderno. Si la ruta no trae .venv,
    #    el kernel no es el del proyecto: Kernel -> Change Kernel.
    ejecutable = Path(sys.executable)
    en_venv = ".venv" in ejecutable.parts or sys.prefix != sys.base_prefix
    # se muestra y se guarda la ruta desde .venv, no la del equipo
    ejecutable_rel = (Path(*ejecutable.parts[ejecutable.parts.index(".venv"):])
                      if ".venv" in ejecutable.parts else ejecutable)
    reporte["interprete"] = str(ejecutable_rel)
    reporte["entorno_virtual"] = bool(en_venv)
    print("Intérprete       :", ejecutable_rel)
    print("Entorno virtual  :", "[OK] activo" if en_venv else "[AVISO] parece el Python del sistema")

    # 2. Carpeta de trabajo: las rutas relativas se resuelven desde acá,
    #    no desde donde quedó guardado el .ipynb.
    reporte["carpeta_trabajo"] = Path.cwd().name
    print("Carpeta de trabajo:", Path.cwd().name)
    print("Sistema          :", platform.system(), platform.release())
    print("Python           :", sys.version.split()[0])

    # 3. Las librerías del proyecto, con su versión.
    print("\nLibrerías declaradas")
    versiones = {}
    for nombre in dependencias:
        try:
            modulo = importlib.import_module(nombre)
            version = getattr(modulo, "__version__", "sin atributo __version__")
            versiones[nombre] = version
            print(f"  [OK]    {nombre:12} {version}")
        except ImportError:
            # No se frena el cuaderno: se avisa qué instalar y con qué nombre.
            versiones[nombre] = None
            paquete = NOMBRE_EN_PYPI.get(nombre, nombre)
            print(f"  [FALTA] {nombre:12} instale con: python -m pip install {paquete}")
    reporte["versiones"] = versiones
    return reporte


ENTORNO = verificar_entorno(DEPENDENCIAS)

Intérprete       : .venv/bin/python
Entorno virtual  : [OK] activo
Carpeta de trabajo: F1
Sistema          : Linux 7.0.0-43-generic
Python           : 3.14.4

Librerías declaradas
  [OK]    numpy        2.5.4
  [OK]    pandas       3.1.1
  [OK]    matplotlib   3.11.2


  [OK]    seaborn      0.13.2
  [OK]    scipy        1.18.1


Si falta una dependencia, debe instalarse desde `requirements.txt` en el mismo entorno que utiliza el kernel.

## 3. Estructura y configuración
Se verifican las carpetas y los archivos vigentes. Ejecutar F1 no sobrescribe código, dependencias ni documentación de las fases posteriores.

In [6]:
def localizar_raiz(inicio=None):
    """
    Devuelve la raíz del proyecto.

    Jupyter abre el kernel en la carpeta donde quedó guardado el cuaderno, no en
    la carpeta desde donde se lanzó JupyterLab. Como cada cuaderno vive en la
    carpeta de su fase (F1, F2, F3 o F4), cuando la carpeta de trabajo es una de
    esas la raíz es la carpeta de arriba. Si el cuaderno corre desde la raíz, se
    usa tal cual.
    """
    inicio = Path.cwd() if inicio is None else Path(inicio)
    return inicio.parent if inicio.name in {"F1", "F2", "F3", "F4"} else inicio


RAIZ = localizar_raiz()

# Rutas de datos COMUNES a todas las fases: las mismas que usa el
# cuaderno de la Fase 2, para que un mismo archivo le sirva a los dos.
DIR_CRUDO = RAIZ / "data" / "raw"          # datos originales: nunca se modifican
DIR_PROCESADO = RAIZ / "data" / "processed"  # salida del pipeline de la Fase 2
DIR_DOCS = RAIZ / "docs"                   # diccionario, fichas y metadatos
DIR_SRC = RAIZ / "src"                     # módulos propios reutilizables

# Una carpeta por fase, para cuadernos e informes.
DIR_FASES = [RAIZ / f"F{n}" for n in (1, 2, 3, 4)]

for carpeta in [DIR_CRUDO, DIR_PROCESADO, DIR_DOCS, DIR_SRC, *DIR_FASES]:
    # parents=True crea las intermedias; exist_ok=True evita el error si ya están,
    # que es lo que deja correr el cuaderno completo de nuevo sin que se rompa.
    carpeta.mkdir(parents=True, exist_ok=True)

print("Estructura creada bajo:", RAIZ.name)
# Se saltan las carpetas ocultas (.git, .venv, .ipynb_checkpoints) y las cachés.
for ruta in sorted(p.relative_to(RAIZ) for p in RAIZ.rglob("*") if p.is_dir()):
    if any(parte.startswith(".") or parte == "__pycache__" for parte in ruta.parts):
        continue
    print("  ", ruta.as_posix() + "/")

Estructura creada bajo: proyecto-grupo4-mcdi500


   F1/
   F2/
   F3/
   F4/
   F4/figuras/
   Material_Didactico_Semana_3/
   Material_Didactico_Semana_3/1_Presentacion_Semana3/
   Material_Didactico_Semana_3/2_Parte_Practica/
   Material_Didactico_Semana_3/2_Parte_Practica/data/
   Material_Didactico_Semana_3/3_Guias_de_Apoyo/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/F1/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/F2/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/F3/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/F4/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/F4/figuras/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/data/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/data/processed/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/data/raw/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/docs/
   Material_Didactico_Semana_3/proyecto-grupoX-mcdi500/src/
   data/
   data/processed/
   data/raw/


Los datos son compartidos por las cuatro fases. La descarga original queda fuera de Git; se utiliza una muestra estratificada por comuna sin nombres, enlaces ni comentarios.

### 3.1 Archivos excluidos
`.gitignore` excluye el entorno, cachés y descargas con datos personales. La muestra de trabajo se conserva en `data/raw/listings.csv`.

In [7]:
ARCHIVO_GITIGNORE = RAIZ / ".gitignore"
lineas = ARCHIVO_GITIGNORE.read_text(encoding="utf-8").splitlines()
assert ".venv/" in lineas
print(".gitignore verificado; no se modifica desde F1.")

.gitignore verificado; no se modifica desde F1.


Ignorar un archivo no lo elimina del historial. La revisión de privacidad debe considerar también las versiones anteriores.

### 3.2 Dependencias
Se comparan las versiones instaladas con las declaradas en `requirements.txt`. No se instala ni actualiza software desde el cuaderno.

In [8]:
ARCHIVO_REQUIREMENTS = RAIZ / "requirements.txt"
requisitos = [l.strip() for l in ARCHIVO_REQUIREMENTS.read_text().splitlines() if l.strip() and not l.startswith("#")]
for requisito in requisitos:
    paquete, declarada = requisito.split("==")
    observada = importlib.metadata.version(paquete)
    assert observada == declarada, f"{paquete}: instalada {observada}; declarada {declarada}"
print("Versiones verificadas:")
print("\n".join(requisitos))

Versiones verificadas:
ipykernel==7.4.0
ipython==9.18.1
jupyterlab==4.6.6
matplotlib==3.11.2
notebook==7.6.5
numpy==2.5.4
pandas==3.1.1
nbclient==0.11.0
nbconvert==7.17.2
nbformat==5.11.1
scipy==1.18.1
seaborn==0.13.2


Se declaran las bibliotecas utilizadas directamente por el proyecto, con versiones fijas.

## 4. Utilidades compartidas
`src/utilidades.py` contiene normalización de nombres, seudonimización y aplanado de metadatos. Se importa desde el paquete instalado sin modificar `sys.path`.

In [9]:
ARCHIVO_MODULO = DIR_SRC / "utilidades.py"
assert ARCHIVO_MODULO.exists()
print("Módulo compartido:", ARCHIVO_MODULO.relative_to(RAIZ))

Módulo compartido: src/utilidades.py


La instalación editable (`python -m pip install -e .`) permite importar `src` desde las carpetas de los cuadernos.

In [10]:
from src import utilidades
print("Importado:", utilidades.__name__)
print(utilidades.normalizar_columnas(["Bathrooms Text", "Property Type", "Room Type"]))

Importado: src.utilidades
['bathrooms_text', 'property_type', 'room_type']


### 4.1 Pruebas
Se comprueban entradas normales, límites y errores de tipo o nombres duplicados.

In [11]:
# --- Caso normal ---
assert utilidades.normalizar_nombre("Bathrooms Text") == "bathrooms_text"
assert utilidades.normalizar_columnas(["A B", "c-d"]) == ["a_b", "c_d"]
print("[OK] Caso normal: normalización correcta")

# --- Caso limite: signos seguidos y espacios en los bordes ---
assert utilidades.normalizar_nombre("  ¿Host ID (anfitrión)?  ") == "host_id_anfitri_n"
assert utilidades.normalizar_columnas([]) == []
print("[OK] Caso límite: signos y lista vacía")

# --- Seudonimización: determinista y distintiva ---
h1 = utilidades.seudonimizar(224592)
assert utilidades.seudonimizar(224592) == h1, "Mismo valor, distinto hash: falla la trazabilidad"
assert utilidades.seudonimizar(244792) != h1, "Dos anfitriones no deben colisionar"
assert len(h1) == 12 and "224592" not in h1, "El código no debe exponer el identificador"
print("[OK] Seudonimización: estable, distintiva y sin el valor original")

# --- Excepcion 1: tipo que no es ---
try:
    utilidades.normalizar_nombre(42)
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (tipo): {e}")

# --- Excepción 2: la normalización colapsa dos nombres en uno ---
# "Beds" y " beds " dan los dos "beds": el módulo lo detecta en vez de perder una columna.
try:
    utilidades.normalizar_columnas(["Beds", " beds "])
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (duplicados): {e}")

# --- Excepcion 3: seudonimizar un tipo no soportado ---
try:
    utilidades.seudonimizar(4.5)
except utilidades.ProyectoError as e:
    print(f"[OK] Excepción capturada (seudónimo): {e}")

[OK] Caso normal: normalización correcta
[OK] Caso límite: signos y lista vacía
[OK] Seudonimización: estable, distintiva y sin el valor original
[OK] Excepción capturada (tipo): Se esperaba texto y se recibio int.
[OK] Excepción capturada (duplicados): La normalizacion dejo nombres duplicados.
[OK] Excepción capturada (seudónimo): Se esperaba texto o entero y se recibio float.


`ProyectoError` permite distinguir los errores de validación de las utilidades.

## 5. Conjunto de datos
La ficha registra la fuente y la versión de la captura. La evaluación revisa estructura, tipos y calidad antes del preprocesamiento.

### 5.1 Procedencia

| Campo | Valor |
| --- | --- |
| Título | Inside Airbnb: Santiago, Chile (`listings.csv` detallado) |
| Autor | Inside Airbnb |
| Plataforma | Inside Airbnb, sección *Get the data* |
| Enlace | `http://insideairbnb.com/get-the-data/` |
| Captura | 25–26 de septiembre de 2026 (`last_scraped`) |
| Estructura original | 18.904 filas × 90 columnas (~50 MB, contiene datos personales: no se versiona) |
| Archivo de trabajo | Muestra estratificada por comuna **anonimizada**: 5.106 filas × 78 columnas (`src/generar_muestra.py`, semilla 42) |
| Unidad de observación | Un alojamiento publicado en Airbnb Santiago |
| Licencia | Datos públicos con fines no comerciales, declarados en la plataforma |

**Referencia en APA 7 para el informe:**

> Inside Airbnb. (2026). *Get the data: Santiago, Chile* [Conjunto de datos]. http://insideairbnb.com/get-the-data/

La muestra se obtiene por comuna con semilla fija (D8). Los nombres y enlaces se eliminan y `host_id` se seudonimiza durante su generación (D5).


In [12]:
FICHA = {
    "titulo": "Inside Airbnb - Santiago, Chile (listings detallado, muestra estratificada)",
    "autor": "Inside Airbnb",
    "plataforma": "Inside Airbnb - Get the data",
    "url": "http://insideairbnb.com/get-the-data/",
    "licencia_abierta": True,
    "unidad_observacion": "Un alojamiento publicado",
    "filas": 5106,
    "columnas": 78,
    "tamano_mb": 8.8,
    # Conteo de variables por ROL ANALITICO (clasificar de validar_dataset.py),
    # no por tipo de dato: el rol es el que manda en el preprocesamiento.
    "roles": {
        "continua": 17,         # latitude, review_scores_*, estimated_revenue...
        "discreta": 27,         # accommodates, availability_*, number_of_reviews...
        "binaria": 5,           # host_is_superhost, has_availability...
        "nominal": 3,           # room_type, neighbourhood_cleansed, host_has_profile_pic
        "ordinal": 0,
        "fecha": 4,             # last_scraped, first_review, last_review, calendar_last_scraped
        "alta_cardinalidad": 8,  # amenities, property_type, host_id seudonimizado
        "identificador": 1,     # id
        "vacia": 13,            # columnas 100% nulas: se eliminan en la Fase 2
    },
    # % máximo de nulos entre variables NO vacías (license); y el máximo absoluto
    # (las 14 columnas vacías): el primero acredita faltantes reales y el
    # segundo deja el aviso que resuelve la Fase 2.
    "pct_nulos_max_no_vacia": 99.0,
    "pct_nulos_max_variable": 100.0,
}

### 5.2 Criterios de selección
La evaluación contrasta la ficha con los requisitos del curso y registra las advertencias de calidad.

In [13]:
# Requisitos del curso. Los OBLIGATORIOS descalifican el conjunto si no se
# cumplen; los AVISOS no, pero lo que falte se declara en el informe como
# restricción del conjunto. Son los mismos que aplica validar_dataset.py.
OBLIGATORIOS = {
    "filas": ("Filas", 2000, "Permite agrupar por categoría sin grupos vacíos"),
    "columnas": ("Columnas", 12, "Asegura variedad de roles analíticos"),
    "roles": ("Roles analíticos distintos", 3, "El conjunto debe ser multivariado"),
    "numerica": ("Variables numéricas", 1, "Escalamiento y valores atípicos"),
    "categorica": ("Variables categóricas", 1, "Codificación de categorías"),
}
AVISOS = {
    "fecha": ("Fecha", 1, "Parseo y derivación de variables temporales"),
    "alta_cardinalidad": ("Texto o alta cardinalidad", 1, "Normalización y agrupación"),
}

# Roles que cuentan para cada grupo. El identificador y las vacías no son
# roles analíticos: el primero solo distingue filas y las otras no traen nada.
NUMERICOS = ("continua", "discreta")
CATEGORICOS = ("binaria", "nominal", "ordinal")
SIN_ROL = ("identificador", "vacia")


def evaluar_criterios(ficha, obligatorios=OBLIGATORIOS, avisos=AVISOS):
    """
    Compara una ficha de conjunto de datos con los requisitos del curso.

    Parametros
    ----------
    ficha : dict
        Ficha del conjunto, con las claves 'filas', 'columnas' y 'roles'.
    obligatorios, avisos : dict
        Criterio -> (etiqueta, minimo, motivo).

    Retorna
    -------
    pd.DataFrame
        Una fila por criterio, con su tipo, el valor observado y el veredicto.

    Lanza
    -----
    KeyError
        Si la ficha no declara los roles analiticos.
    """
    if "roles" not in ficha:
        raise KeyError("La ficha debe declarar el conteo de variables por rol analítico.")

    roles = ficha["roles"]
    observado = {
        "filas": ficha.get("filas", 0),
        "columnas": ficha.get("columnas", 0),
        # Cuántos roles distintos salen al menos una vez, sin contar los
        # que no aportan al análisis (identificador y vacías).
        "roles": sum(1 for rol, n in roles.items() if n > 0 and rol not in SIN_ROL),
        "numerica": sum(roles.get(r, 0) for r in NUMERICOS),
        "categorica": sum(roles.get(r, 0) for r in CATEGORICOS),
        "fecha": roles.get("fecha", 0),
        "alta_cardinalidad": roles.get("alta_cardinalidad", 0),
    }

    filas = []
    for tipo, criterios in (("obligatorio", obligatorios), ("aviso", avisos)):
        for clave, (etiqueta, minimo, motivo) in criterios.items():
            valor = observado[clave]
            # Un aviso incumplido no dice NO: dice que hay algo por declarar.
            falla = "NO" if tipo == "obligatorio" else "aviso"
            filas.append({
                "criterio": etiqueta,
                "tipo": tipo,
                "mínimo": minimo,
                "observado": valor,
                "cumple": "sí" if valor >= minimo else falla,
                "por qué se pide": motivo,
            })

    # Los faltantes se informan como aviso: un conjunto sin ellos sirve igual,
    # pero deja con poca evidencia la sección de preprocesamiento de la Fase 2.
    pct = ficha.get("pct_nulos_max_no_vacia", 0.0)
    filas.append({
        "criterio": "Alguna variable con ≥1% de nulos", "tipo": "aviso",
        "mínimo": 1.0, "observado": pct,
        "cumple": "sí" if pct >= 1.0 else "aviso",
        "por qué se pide": "Un conjunto perfecto no tiene preprocesamiento que justificar",
    })
    pct_max = ficha.get("pct_nulos_max_variable", 0.0)
    filas.append({
        "criterio": "Ninguna variable sobre 60% de nulos", "tipo": "aviso",
        "mínimo": 60.0, "observado": pct_max,
        "cumple": "sí" if pct_max <= 60.0 else "aviso",
        "por qué se pide": "Por encima de eso la variable no es recuperable",
    })
    tabla = pd.DataFrame(filas)
    for col in ("mínimo", "observado"):
        tabla[col] = tabla[col].map(lambda v: f"{v:g}")
    return tabla


evaluacion = evaluar_criterios(FICHA)
evaluacion

,criterio,tipo,mínimo,observado,cumple,por qué se pide
0,Filas,obligatorio,2000,5106,sí,Permite agrupar por categoría sin grupos vacíos
1,Columnas,obligatorio,12,78,sí,Asegura variedad de roles analíticos
2,Roles analíticos distintos,obligatorio,3,6,sí,El conjunto debe ser multivariado
3,Variables numéricas,obligatorio,1,44,sí,Escalamiento y valores atípicos
4,Variables categóricas,obligatorio,1,8,sí,Codificación de categorías
5,Fecha,aviso,1,4,sí,Parseo y derivación de variables temporales
6,Texto o alta cardinalidad,aviso,1,8,sí,Normalización y agrupación
7,Alguna variable con ≥1% de nulos,aviso,1,99,sí,Un conjunto perfecto no tiene preprocesamiento...
8,Ninguna variable sobre 60% de nulos,aviso,60,100,aviso,Por encima de eso la variable no es recuperable


In [14]:
incumplidos = evaluacion.loc[evaluacion["cumple"] == "NO", "criterio"].tolist()
avisos = evaluacion.loc[evaluacion["cumple"] == "aviso", "criterio"].tolist()
print(f"Criterios evaluados : {len(evaluacion)}")
print(f"Obligatorios sin cumplir: {len(incumplidos)}")
print(f"Avisos              : {len(avisos)}")
if incumplidos:
    print("\nNo cumple (descalifica el conjunto):")
    for c in incumplidos:
        print("  ·", c)
if avisos:
    print("\nAvisos (no descalifican; se declaran en el informe):")
    for c in avisos:
        print("  ·", c)

Criterios evaluados : 9
Obligatorios sin cumplir: 0
Avisos              : 1

Avisos (no descalifican; se declaran en el informe):
  · Ninguna variable sobre 60% de nulos


Las columnas vacías requieren tratamiento en F2. Las cifras observadas se contrastan con el archivo de trabajo, no solo con la ficha.

### 5.3 Diccionario
Se distingue el rol analítico del tipo almacenado: precio como texto requiere conversión e identificadores numéricos no son variables de magnitud.

In [15]:
DICCIONARIO = [
    ("id",                        "identificador", "Identificador del alojamiento; une con reseñas"),
    ("host_id",                   "identificador", "Identificador del anfitrión; se seudonimiza (D5)"),
    ("neighbourhood_cleansed",    "nominal",       "Comuna (32 categorías)"),
    ("latitude",                  "continua",      "Latitud del alojamiento"),
    ("longitude",                 "continua",      "Longitud del alojamiento"),
    ("property_type",             "alta cardinalidad", "Tipo de propiedad (70 categorías; se agrupan en F2)"),
    ("room_type",                 "nominal",       "Entire home/apt, Private room, Hotel room, Shared room"),
    ("accommodates",              "discreta",      "Capacidad de huéspedes"),
    ("bedrooms",                  "discreta",      "Número de dormitorios"),
    ("beds",                      "discreta",      "Número de camas"),
    ("bathrooms_text",            "nominal",       "Baños en texto: '1 bath', '1.5 shared baths'..."),
    ("amenities",                 "texto libre",   "Lista JSON de comodidades; se binariza en F2"),
    ("price",                     "continua",      "Precio por noche; viene como texto '$59,303.00'"),
    ("minimum_nights",            "discreta",      "Noches mínimas de estadía"),
    ("availability_365",          "discreta",      "Días disponibles en el próximo año; define 'activo'"),
    ("number_of_reviews",         "discreta",      "Reseñas acumuladas"),
    ("number_of_reviews_ltm",     "discreta",      "Reseñas últimos 12 meses; define 'activo'"),
    ("number_of_reviews_l30d",    "discreta",      "Reseñas últimos 30 días"),
    ("reviews_per_month",         "continua",      "Reseñas por mes; nula cuando no hay reseñas"),
    ("estimated_occupancy_l365d", "discreta",      "Ocupación estimada últimos 365 días; tope 255 noches"),
    ("estimated_revenue_l365d",   "continua",      "Ingresos estimados = ocupación × precio (derivada)"),
    ("first_review",              "fecha",         "Fecha de la primera reseña"),
    ("last_review",               "fecha",         "Fecha de la última reseña"),
    ("review_scores_rating",      "continua",      "Puntaje global; nulo cuando no hay reseñas"),
    ("host_is_superhost",         "binaria",       "t/f: anfitrión con distinción superhost"),
    ("calculated_host_listings_count", "discreta", "Anuncios del mismo anfitrión"),
]

diccionario = pd.DataFrame(DICCIONARIO,
                           columns=["variable", "rol_analitico", "descripcion"])

# Verificación: las variables declaradas tienen que existir en el archivo real.
assert diccionario["rol_analitico"].str.strip().ne("").all(), "Hay roles vacíos."
print(f"{len(diccionario)} variables declaradas de las {FICHA['columnas']} del archivo.")
print("(El rol observado de las 90 columnas se documenta en la Fase 2.)\n")
print(diccionario["rol_analitico"].value_counts().to_string())
diccionario

26 variables declaradas de las 78 del archivo.
(El rol observado de las 90 columnas se documenta en la Fase 2.)

rol_analitico
discreta             10
continua              6
nominal               3
identificador         2
fecha                 2
alta cardinalidad     1
texto libre           1
binaria               1


,variable,rol_analitico,descripcion
0,id,identificador,Identificador del alojamiento; une con reseñas
1,host_id,identificador,Identificador del anfitrión; se seudonimiza (D5)
2,neighbourhood_cleansed,nominal,Comuna (32 categorías)
3,latitude,continua,Latitud del alojamiento
4,longitude,continua,Longitud del alojamiento
5,property_type,alta cardinalidad,Tipo de propiedad (70 categorías; se agrupan e...
6,room_type,nominal,"Entire home/apt, Private room, Hotel room, Sha..."
7,accommodates,discreta,Capacidad de huéspedes
8,bedrooms,discreta,Número de dormitorios
9,beds,discreta,Número de camas


El precio y los baños requieren conversión. La ocupación y los ingresos se conservan como estimaciones de la fuente, con sus limitaciones documentadas.

### 5.4 Comprobación de la muestra
Se compara el archivo disponible con las dimensiones y variables declaradas.

In [16]:
ARCHIVO = DIR_CRUDO / "listings.csv"

if ARCHIVO.exists():
    df_crudo = pd.read_csv(ARCHIVO)
    print("Archivo encontrado :", ARCHIVO.relative_to(RAIZ).as_posix())
    print("Dimensiones reales :", df_crudo.shape)
    print("Declarado en ficha :", (FICHA["filas"], FICHA["columnas"]))
    coincide = df_crudo.shape == (FICHA["filas"], FICHA["columnas"])
    print("Coincidencia       :", "[OK]" if coincide else "[REVISAR] ficha y archivo difieren")

    # Las variables del diccionario tienen que estar en el archivo.
    ausentes = [v for v in diccionario["variable"] if v not in df_crudo.columns]
    print("Variables del diccionario ausentes en el archivo:",
          ausentes if ausentes else "ninguna [OK]")

    print("\nNulos relevantes (columnas con entre 1 y 99 % de faltantes):")
    nulos = (df_crudo.isna().mean() * 100).round(1)
    print(nulos[(nulos > 0) & (nulos < 99)].sort_values(ascending=False).to_string())
    print(f"\nColumnas 100 % vacías: {int((df_crudo.isna().mean() == 1).sum())}")
else:
    df_crudo = None
    print("[PENDIENTE] El archivo aún no está en", DIR_CRUDO.relative_to(RAIZ).as_posix())
    print("1. Descargue 'listings.csv' (detallado) desde:", FICHA["url"])
    print("2. Genere la muestra:  python src/generar_muestra.py ruta/al/listings.csv")
    print("\nLa Fase 1 no requiere el archivo: la exploración es trabajo de la Fase 2.")

Archivo encontrado : data/raw/listings.csv
Dimensiones reales : (5106, 78)
Declarado en ficha : (5106, 78)
Coincidencia       : [OK]
Variables del diccionario ausentes en el archivo: ninguna [OK]

Nulos relevantes (columnas con entre 1 y 99 % de faltantes):
source                         95.3
review_scores_communication    14.7
review_scores_location         14.7
last_review                    14.7
first_review                   14.7
review_scores_value            14.7
reviews_per_month              14.7
review_scores_accuracy         14.7
review_scores_rating           14.7
review_scores_checkin          14.7
review_scores_cleanliness      14.7
price_quote_price_per_night     4.7
price                           4.7
estimated_revenue_l365d         4.7
price_quote_total_price         4.7
price_quote_checkin_date        3.2
price_quote_checkout_date       3.2
price_quote_raw                 3.2
beds                            1.1
has_availability                0.9
bathrooms_text        

## 6. Control de versiones
Las consultas siguientes muestran el estado del repositorio; no crean commits ni modifican la configuración de Git.

In [17]:
def git(*argumentos):
    """
    Ejecuta un comando de Git y devuelve su salida como texto.

    Retorna
    -------
    (bool, str)
        Exito de la ejecucion y salida (o mensaje de error).
    """
    if shutil.which("git") is None:
        return False, "Git no está instalado o no está en el PATH."
    try:
        proceso = subprocess.run(["git", *argumentos], capture_output=True,
                                 text=True, timeout=20, cwd=RAIZ)
    except (OSError, subprocess.SubprocessError) as e:
        return False, f"No se pudo ejecutar Git: {e}"
    if proceso.returncode != 0:
        # stderr trae el motivo real; se devuelve tal cual para no esconderlo.
        return False, proceso.stderr.strip() or "Git terminó con error."
    return True, proceso.stdout.strip()


ok, salida = git("--version")
print("Git                :", salida if ok else f"[AVISO] {salida}")

for clave in ("user.name", "user.email"):
    ok_cfg, valor = git("config", "--get", clave)
    estado = valor if ok_cfg and valor else "[FALTA] configúrelo con git config --global"
    print(f"{clave:19}:", estado)

Git                : git version 2.53.0
user.name          : Mauricio Martinez
user.email         : mmartinez@whitestack.com


Una identidad consistente facilita atribuir los aportes. No se reescribe el historial desde este cuaderno.

In [18]:
es_repo, _ = git("rev-parse", "--is-inside-work-tree")

if es_repo == "true" or es_repo is True:
    ok_rama, rama = git("rev-parse", "--abbrev-ref", "HEAD")
    print("Rama actual        :", rama if ok_rama else "sin commits todavía")

    ok_log, log = git("log", "--oneline", "-5")
    print("\nÚltimos commits")
    print(log if ok_log and log else "  (todavía no hay commits)")

    # --porcelain da una salida estable, pensada para que la lea un programa.
    ok_st, estado = git("status", "--porcelain")
    pendientes = [l for l in estado.splitlines() if l.strip()] if ok_st else []
    print(f"\nArchivos con cambios sin registrar: {len(pendientes)}")
    for linea in pendientes[:10]:
        print("  ", linea)

    ok_rem, remoto = git("remote", "-v")
    print("\nRemoto             :", remoto.splitlines()[0] if ok_rem and remoto
          else "[FALTA] agregue el remoto con git remote add origin URL")
else:
    print("[PENDIENTE] Esta carpeta todavía no es un repositorio Git.")
    print("Desde la terminal, en la raíz del proyecto:")
    print("  git init")
    print("  git add .gitignore requirements.txt README.md")
    print('  git commit -m "docs: estructura inicial del proyecto"')

Rama actual        : feature/cierre-final-proyecto



Últimos commits
9397168 F4 alfredo valenzuela (#15)
35ae24d chore: Correcciones indicadas (#14)
73da241 feature/censura-sesgos (#12)
ed5962a feat: se corrige patron de diseño en F3, y en F4 se agrega nuevo grafico que muestra la cantidad de alojamientos por comuna, agrupando a los que tienen menos de 30 en categoria otras (#13)
5a9e972 Merge pull request #11 from RedWolfo10/fix/pruebas-automaticas

Archivos con cambios sin registrar: 47
   M F1/S1_F1_Definicion.ipynb
    M F2/S1_F2_Preprocesamiento.ipynb
    M F3/S2_F3_NucleoAlgoritmico_Eficiencia_POO.ipynb
    M F4/S3_F4_Visualizacion_Comunicacion.ipynb
    M F4/figuras/fig1_cantidad_alojamientos_comuna.png
    M F4/figuras/fig1_ocupacion_comuna.png
    M F4/figuras/fig2_precio_ocupacion_comuna.png
    M F4/figuras/fig3_desempeno_dormitorios.png
    M F4/figuras/fig3b_ingreso_huesped_dormitorios.png
    M F4/figuras/fig3c_caja_ocupacion_dormitorios.png

Remoto             : origin	git@github.com:RedWolfo10/proyecto-grupo4-mcdi500.git

Los mensajes usan prefijos como `feat`, `fix`, `data` y `docs`, seguidos de una descripción concreta del cambio.

## 7. Validación
Se comprueban estructura, archivos de configuración, definición del problema, variables e importación del módulo.

In [19]:
def validar_fase1(raiz, config, ficha, diccionario, df_archivo=None):
    """
    Comprueba que la Fase 1 dejo lo que la rubrica exige.
    Lanza AssertionError si alguna comprobacion falla.
    """
    print("VALIDACION DE LA FASE 1")
    print("-" * 45)

    # 1. Estructura de carpetas
    esperadas = ["data/raw", "data/processed", "docs", "src", "F1", "F2", "F3", "F4"]
    faltan = [c for c in esperadas if not (raiz / c).is_dir()]
    assert not faltan, f"Faltan carpetas: {faltan}"
    print(f"[OK] Estructura completa ({len(esperadas)} carpetas)")

    # 2. Artefactos del entorno reproducible
    for nombre in (".gitignore", "requirements.txt"):
        archivo = raiz / nombre
        assert archivo.exists(), f"Falta {nombre}."
        assert archivo.stat().st_size > 0, f"{nombre} está vacío."
        print(f"[OK] {nombre} presente ({archivo.stat().st_size} bytes)")

    # 3. Definición del problema
    for clave in ("titulo", "problematica", "objetivo_general", "objetivos_especificos"):
        assert config.get(clave), f"La configuración no declara '{clave}'."
    assert len(config["objetivos_especificos"]) >= 3, "Se esperan al menos tres objetivos."
    print("[OK] Definición del problema completa")

    # 4. Coherencia del diccionario: roles declarados y variables existentes
    assert len(diccionario) >= 15, "El diccionario debe cubrir las variables del análisis."
    roles_declarados = set(diccionario["rol_analitico"])
    assert roles_declarados, "El diccionario no declara roles analíticos."
    if df_archivo is not None:
        ausentes = set(diccionario["variable"]) - set(df_archivo.columns)
        assert not ausentes, f"Variables del diccionario ausentes del archivo: {ausentes}"
        print(f"[OK] Diccionario coherente ({len(diccionario)} variables declaradas "
              f"de {df_archivo.shape[1]}, {len(roles_declarados)} roles)")
    else:
        print(f"[OK] Diccionario declarado ({len(diccionario)} variables, "
              f"{len(roles_declarados)} roles; archivo pendiente)")

    # 5. Módulo importable
    assert (raiz / "src" / "utilidades.py").exists(), "Falta el módulo src/utilidades.py."
    assert callable(getattr(utilidades, "normalizar_columnas", None)), \
        "El módulo no expone normalizar_columnas."
    assert callable(getattr(utilidades, "seudonimizar", None)), \
        "El módulo no expone seudonimizar."
    print("[OK] Módulo src/utilidades.py importable")

    print(f"\n[INFO] Conjunto declarado: {ficha['titulo']}")
    print(f"[INFO] Archivo generado: {'sí' if ARCHIVO.exists() else 'todavía no'}")
    return True


validar_fase1(RAIZ, PROYECTO, FICHA, diccionario, df_crudo)

VALIDACION DE LA FASE 1
---------------------------------------------
[OK] Estructura completa (8 carpetas)
[OK] .gitignore presente (806 bytes)
[OK] requirements.txt presente (195 bytes)
[OK] Definición del problema completa
[OK] Diccionario coherente (26 variables declaradas de 78, 8 roles)
[OK] Módulo src/utilidades.py importable

[INFO] Conjunto declarado: Inside Airbnb - Santiago, Chile (listings detallado, muestra estratificada)
[INFO] Archivo generado: sí


True

## 8. Mapa conceptual
La tabla vincula los conceptos del curso con archivos y resultados verificables del proyecto.

In [20]:
VINCULACION = [
    # (nodo del mapa, tipo, estado, dónde se implementa, evidencia verificable)
    ("Entorno de desarrollo en Python",       "herramienta", "Implementado", "Sección 2 · verificar_entorno", "Intérprete .venv y versiones impresas"),
    ("Ambiente virtual (.venv)",              "herramienta", "Implementado", "Sección 2 · verificar_entorno", "Ruta del intérprete contiene .venv"),
    ("requirements.txt",                      "flujo reproducible", "Implementado", "Sección 3.2 · escribir_requirements", "Archivo generado desde el entorno"),
    ("Dependencias con versión fija",         "flujo reproducible", "Implementado", "Sección 3.2 · requirements.txt", "requirements.txt en la raíz"),
    ("Git y GitHub",                          "herramienta", "Implementado", "Sección 6 · consultas a Git", "Rama, commits, remoto y correo"),
    ("Commits descriptivos y ramas",          "organización del repositorio", "Implementado", "Sección 6 · convención de commits", "Historial con prefijos docs/feat/fix"),
    ("Repositorio del proyecto",              "organización del repositorio", "Implementado", "Sección 3 · localizar_raiz y árbol", "Estructura impresa y validada"),
    ("Carpetas F1 y F2",                      "organización del repositorio", "Implementado", "Sección 3 · DIR_FASES", "Carpetas F1-F4 creadas"),
    ("Jupyter Notebook",                      "herramienta", "Implementado", "Este cuaderno", "F1/S1_F1_Definicion.ipynb ejecutado"),
    ("NumPy y Pandas",                        "herramienta", "Implementado", "Secciones 5 y 9 · tablas de documentación", "Versiones verificadas y DataFrames"),
    ("Funciones, módulos y scripts reutilizables", "proceso", "Implementado", "Sección 4 · src/utilidades.py", "Módulo importado y probado"),
    ("Documentación en Markdown y comentarios", "estructura del notebook", "Implementado", "Celdas narrativas y docstrings", "Cuaderno legible, docs/ generado"),
    ("README del proyecto",                   "organización del repositorio", "Implementado", "Sección 9 · generar_readme", "README.md en la raíz"),
    ("Validación técnica y pruebas",          "proceso", "Implementado", "Secciones 4.1 y 7 · asserts", "Casos normal, límite y excepciones"),
    ("Datos públicos de Inside Airbnb",       "proceso", "Implementado", "Sección 5 · FICHA y procedencia", "Ficha, cita APA 7 y evaluación"),
    ("Preparación y análisis de los datos",   "flujo reproducible", "Implementado", "Muestra estratificada + Fase 2", "src/generar_muestra.py y F2/"),
    # Proyectados a fases siguientes: la rubrica pide declararlos tambien.
    ("Inferencia del idioma de las reseñas",  "proceso", "Proyectado (F3)", "-", "Decisión D6 pendiente; requiere reviews.csv.gz"),
    ("Indicadores por comuna y configuración","proceso", "Proyectado (F3)", "-", "Objetivos específicos 3 y 4"),
    ("Visualizaciones e informe final",       "herramienta", "Proyectado (F4)", "-", "Matplotlib/Seaborn + informe técnico"),
    ("Escalamiento de variables (z-score)",   "decisión técnica", "Descartado", "Cuaderno F2 · sección 6", "Proyecto descriptivo: no aplica"),
]

vinculacion = pd.DataFrame(
    VINCULACION,
    columns=["nodo_mapa", "tipo", "estado", "donde_se_implementa", "evidencia"])

# Ningún nodo del mapa puede quedar sin evidencia: eso es lo que revisa la rúbrica.
assert vinculacion["evidencia"].str.strip().ne("").all(), "Hay nodos sin evidencia."
n_impl = int((vinculacion["estado"] == "Implementado").sum())
print(f"{len(vinculacion)} nodos del mapa conceptual "
      f"({n_impl} implementados, {len(vinculacion) - n_impl} proyectados o descartados).\n")
vinculacion


20 nodos del mapa conceptual (16 implementados, 4 proyectados o descartados).



,nodo_mapa,tipo,estado,donde_se_implementa,evidencia
0,Entorno de desarrollo en Python,herramienta,Implementado,Sección 2 · verificar_entorno,Intérprete .venv y versiones impresas
1,Ambiente virtual (.venv),herramienta,Implementado,Sección 2 · verificar_entorno,Ruta del intérprete contiene .venv
2,requirements.txt,flujo reproducible,Implementado,Sección 3.2 · escribir_requirements,Archivo generado desde el entorno
3,Dependencias con versión fija,flujo reproducible,Implementado,Sección 3.2 · requirements.txt,requirements.txt en la raíz
4,Git y GitHub,herramienta,Implementado,Sección 6 · consultas a Git,"Rama, commits, remoto y correo"
5,Commits descriptivos y ramas,organización del repositorio,Implementado,Sección 6 · convención de commits,Historial con prefijos docs/feat/fix
6,Repositorio del proyecto,organización del repositorio,Implementado,Sección 3 · localizar_raiz y árbol,Estructura impresa y validada
7,Carpetas F1 y F2,organización del repositorio,Implementado,Sección 3 · DIR_FASES,Carpetas F1-F4 creadas
8,Jupyter Notebook,herramienta,Implementado,Este cuaderno,F1/S1_F1_Definicion.ipynb ejecutado
9,NumPy y Pandas,herramienta,Implementado,Secciones 5 y 9 · tablas de documentación,Versiones verificadas y DataFrames


Cada vínculo identifica una implementación o una evidencia disponible en el repositorio.

## 9. Registro de la fase
Se conservan las tablas y los metadatos de ejecución. El README se mantiene como documentación compartida de las cuatro fases.

In [21]:
ARCHIVO_README = RAIZ / "README.md"
contenido = ARCHIVO_README.read_text(encoding="utf-8")
assert contenido.strip()
print("README conservado:", len(contenido.splitlines()), "líneas")

README conservado: 103 líneas


In [22]:
# Artefactos de documentación de la fase, en formato de tabla para el informe.
evaluacion.to_csv(DIR_DOCS / "evaluacion_criterios_dataset.csv", index=False)
vinculacion.to_csv(DIR_DOCS / "vinculacion_mapa_conceptual.csv", index=False)

# Metadatos de la fase: lo que deja reconstruir en qué condiciones corrió.
# El commit corto deja la referencia del punto del historial en que se generó,
# que es la trazabilidad que pide la rúbrica para cada resultado relevante.
ok_head, commit_corto = git("rev-parse", "--short", "HEAD")

METADATOS = {
    "proyecto": PROYECTO["titulo"],
    "grupo": PROYECTO["grupo"],
    "fase": "F1",
    "fecha_ejecucion": date.today().isoformat(),
    "commit": commit_corto if ok_head else "sin commits",
    "semilla": SEMILLA,
    "python": sys.version.split()[0],
    "sistema": f"{platform.system()} {platform.release()}",
    "interprete": Path(sys.executable).name,
    "entorno_virtual": ENTORNO["entorno_virtual"],
    "dependencias": ENTORNO["versiones"],
    "dataset": {"titulo": FICHA["titulo"], "url": FICHA["url"],
                "archivo_generado": ARCHIVO.exists(),
                "filas_muestra": FICHA["filas"]},
}
(DIR_DOCS / "metadatos_fase1.json").write_text(
    json.dumps(METADATOS, indent=2, ensure_ascii=False), encoding="utf-8")

for archivo in sorted(DIR_DOCS.iterdir()):
    print(f"  {archivo.relative_to(RAIZ).as_posix():48} {archivo.stat().st_size:6} bytes")

  docs/comodidades_por_estratos.csv                 46993 bytes
  docs/comparacion_comodidades.csv                   3610 bytes
  docs/configuracion_banos.csv                       1009 bytes
  docs/configuracion_capacidad.csv                    708 bytes
  docs/configuracion_dormitorios.csv                  812 bytes
  docs/conteos_comuna_dormitorios.csv                 153 bytes
  docs/diccionario_variables.csv                     3692 bytes
  docs/eficiencia_f3.csv                              398 bytes
  docs/evaluacion_criterios_dataset.csv               785 bytes
  docs/hallazgos_f4.csv                               855 bytes
  docs/idioma_comunas.csv                             505 bytes
  docs/metadatos_f4.json                             1376 bytes
  docs/metadatos_fase1.json                           711 bytes
  docs/metadatos_fase3.csv                            561 bytes
  docs/metadatos_proyecto.csv                        1120 bytes
  docs/registro_decisiones.md           

In [23]:
# Resumen de cierre: qué había al empezar y qué queda al terminar la fase.
resumen = pd.DataFrame([
    {"artefacto": "Carpetas del proyecto",
     "estado": len([p for p in RAIZ.rglob("*") if p.is_dir()
                    and not any(parte.startswith(".") for parte in p.parts)])},
    {"artefacto": "Archivos en docs/", "estado": len(list(DIR_DOCS.iterdir()))},
    {"artefacto": "Dependencias declaradas", "estado": len(requisitos)},
    {"artefacto": "Variables en el diccionario", "estado": len(diccionario)},
    {"artefacto": "Nodos del mapa vinculados", "estado": len(vinculacion)},
    {"artefacto": "Criterios de dataset incumplidos", "estado": len(incumplidos)},
])
resumen

,artefacto,estado
0,Carpetas del proyecto,31
1,Archivos en docs/,21
2,Dependencias declaradas,12
3,Variables en el diccionario,26
4,Nodos del mapa vinculados,20
5,Criterios de dataset incumplidos,0


## Cierre
La fase documenta el problema, la fuente y el entorno. La muestra y el diccionario sirven como entrada de F2; los metadatos permiten identificar esta ejecución.

## 10. Reflexión
El archivo requiere convertir precio y baños, revisar faltantes y tratar las
comodidades como listas. La muestra facilita la ejecución, pero limita la
extrapolación. Los ingresos dependen de la ocupación y del precio publicado;
no son una medida independiente ni permiten estimar retorno de inversión.

Las decisiones posteriores se consultan en `docs/registro_decisiones.md`.
El idioma se utiliza en F4 solo como descripción exploratoria de las reseñas,
no como indicador de nacionalidad.

## 11. Verificación de cierre
Se ejecuta F1 completo con el entorno instalado y se conservan sus salidas. La comprobación final del proyecto vuelve a ejecutar F1–F4 y las pruebas automáticas.